# 章节实践

本节用于检查第二章的掌握情况。练习会围绕一个最小 PyPTO 算子的开发闭环展开：明确计算目标、编写 Kernel、准备 Host 侧输入输出、设置运行模式并用 PyTorch 结果完成验证。


## 1. 实践目标

完成本节后，读者应能够：

1. 说明 PyPTO Hello World 算子的基本组成。
2. 区分 Host 侧数据准备、Kernel 侧计算表达和验证逻辑。
3. 理解 `@pypto.frontend.jit`、`pypto.Tensor`、`pypto.set_vec_tile_shapes` 等基础接口的作用。
4. 在已有加法算子的基础上完成一个简单表达式改写，并同步调整参考验证。


## 2. 章节实践题

题型包含选择题、填空题和编程题。建议先独立完成，再执行下一单元查看参考答案。

1. （选择题）一个最小 PyPTO 算子的开发闭环通常包含哪组步骤？  
   A. 只安装 Python 包即可，不需要定义计算和验证  
   B. 明确计算目标，定义 Kernel，准备输入输出，调用 Kernel，并用 PyTorch 参考结果验证  
   C. 只创建输出 Tensor，不需要输入 Tensor  
   D. 只写 Host 侧打印语句，不需要 Kernel

2. （填空题）在 PyPTO 示例中，Host 侧通常负责准备输入输出、调用 Kernel 和结果验证；Kernel 侧负责描述________。

3. （选择题）`@pypto.frontend.jit` 的主要作用是什么？  
   A. 标记一段函数作为 PyPTO Kernel，由 PyPTO 前端处理其中的计算表达  
   B. 将 notebook 自动转换为 Markdown  
   C. 删除所有输入 Tensor  
   D. 只用于打印日志

4. （填空题）`pypto.Tensor([1, 4, 1024], pypto.DT_FP32)` 这类声明主要描述 Kernel 参数的________和________。

5. （选择题）在 Hello World 加法示例中，为什么还要编写 PyTorch 参考结果？  
   A. 用参考结果检查 Kernel 输出是否符合预期  
   B. 让 Kernel 不再执行  
   C. 替代所有 PyPTO API  
   D. 只为了增加代码行数

6. （填空题）PyPTO 计算图可以帮助理解从 Tensor 表达逐步进入更低层执行组织的过程，其中本章提到的层次包括 Tensor Graph、Tile Graph、Block Graph 和________。

7. （编程题）实现half_add_kernel方法，实现 `out = (x + y) * 0.5`。要求Kernel 侧完成加法和乘常数，Host 侧代码已给出，只需编写kernel侧代码。
   


In [ ]:
import os
os.environ['TILE_FWK_DEVICE_ID'] = '0'
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'
import pypto
import torch
import torch_npu

RUN_MODE = pypto.RunMode.NPU

def get_device():
    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    return f"npu:{device_id}"


# TODO 此处添加kernel函数的实现


def test_half_add_kernel():
    shape = (64, 64)
    device = get_device()
    x = torch.randn(shape, dtype=torch.float, device=device)
    y = torch.randn(shape, dtype=torch.float, device=device)
    out = torch.empty(shape, dtype=torch.float, device=device)
    half_add_kernel(x, y, out)
    torch.testing.assert_close((x + y) * 0.5, out, atol=1e-3, rtol=1e-3)
    print("✓ Test completed successfully")


test_half_add_kernel()

## 3. 查看答案

执行以下代码获取参考答案。


In [ ]:
!cat ./answer/02.05_answer.py


## 4. 本章小结

第二章完成了 PyPTO 算子开发基础闭环：Kernel 侧描述计算，Host 侧准备数据、调用和验证，PyPTO 通过 JIT、Tensor 声明、Tile Shape 和运行模式把高层表达接入执行流程。进入下一章后，这个闭环会扩展到逐元素、矩阵乘、规约和 shape 处理等更多算子类型。
